In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from scikeras.wrappers import KerasClassifier
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping

In [4]:
data = pd.read_csv('churn.csv')

In [5]:
## Pre-processing
data = data.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1)
data

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0
...,...,...,...,...,...,...,...,...,...,...,...
9995,771,France,Male,39,5,0.00,2,1,0,96270.64,0
9996,516,France,Male,35,10,57369.61,1,1,1,101699.77,0
9997,709,France,Female,36,7,0.00,1,0,1,42085.58,1
9998,772,Germany,Male,42,3,75075.31,2,1,0,92888.52,1


In [6]:
## Encode categorical variables = Geography and Gender
label_encoder_gender = LabelEncoder()
data['Gender'] = label_encoder_gender.fit_transform(data['Gender'])
data

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,0,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,0,41,1,83807.86,1,0,1,112542.58,0
2,502,France,0,42,8,159660.80,3,1,0,113931.57,1
3,699,France,0,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,0,43,2,125510.82,1,1,1,79084.10,0
...,...,...,...,...,...,...,...,...,...,...,...
9995,771,France,1,39,5,0.00,2,1,0,96270.64,0
9996,516,France,1,35,10,57369.61,1,1,1,101699.77,0
9997,709,France,0,36,7,0.00,1,0,1,42085.58,1
9998,772,Germany,1,42,3,75075.31,2,1,0,92888.52,1


In [7]:
from sklearn.preprocessing import OneHotEncoder

one_hot_encoder = OneHotEncoder(sparse_output = False)

one_hot_encoder_geo = one_hot_encoder.fit_transform(data[['Geography']])

one_hot_encoder_geo

array([[1., 0., 0.],
       [0., 0., 1.],
       [1., 0., 0.],
       ...,
       [1., 0., 0.],
       [0., 1., 0.],
       [1., 0., 0.]])

In [8]:
one_hot_encoder.get_feature_names_out(['Geography']) ## the same thing will be used while prediction

array(['Geography_France', 'Geography_Germany', 'Geography_Spain'],
      dtype=object)

In [9]:
geo_one_hot_encoded_columns = pd.DataFrame(one_hot_encoder_geo, columns=['Geography_France', 'Geography_Germany', 'Geography_Spain'])

In [10]:
data = pd.concat([data.drop(['Geography'], axis=1), geo_one_hot_encoded_columns], axis=1)

In [11]:
## divide data into ind. and dep. features
X = data.drop(['Exited'], axis=1)
y = data['Exited']

In [12]:
X_train, X_test, y_train, y_test = train_test_split(
  X, y, test_size = 0.20, random_state = 42
)

In [13]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [14]:
pd.DataFrame(X_train) ## 80% of data after scaling

,0,1,2,3,4,5,6,7,8,9,10,11
0,0.356500,0.913248,-0.655786,0.345680,-1.218471,0.808436,0.649203,0.974817,1.367670,1.001501,-0.579467,-0.576388
1,-0.203898,0.913248,0.294938,-0.348369,0.696838,0.808436,0.649203,0.974817,1.661254,-0.998501,1.725723,-0.576388
2,-0.961472,0.913248,-1.416365,-0.695393,0.618629,-0.916688,0.649203,-1.025834,-0.252807,-0.998501,-0.579467,1.734942
3,-0.940717,-1.094993,-1.131148,1.386753,0.953212,-0.916688,0.649203,-1.025834,0.915393,1.001501,-0.579467,-0.576388
4,-1.397337,0.913248,1.625953,1.386753,1.057449,-0.916688,-1.540351,-1.025834,-1.059600,1.001501,-0.579467,-0.576388
...,...,...,...,...,...,...,...,...,...,...,...,...
7995,1.207474,0.913248,1.435808,1.039728,-0.102301,-0.916688,0.649203,0.974817,-0.539860,1.001501,-0.579467,-0.576388
7996,0.314989,-1.094993,1.816097,-1.389442,-1.218471,-0.916688,0.649203,0.974817,-1.733882,1.001501,-0.579467,-0.576388
7997,0.865009,-1.094993,-0.085351,-1.389442,-1.218471,2.533560,-1.540351,-1.025834,-0.142765,1.001501,-0.579467,-0.576388
7998,0.159323,0.913248,0.390011,1.039728,1.827259,-0.916688,0.649203,-1.025834,-0.050826,1.001501,-0.579467,-0.576388


In [33]:
import pickle

## Save encoders
with open('label_encoder_gender.pkl', 'wb') as file:
  pickle.dump(label_encoder_gender, file)

with open('one_hot_encoder.pkl', 'wb') as file:
  pickle.dump(one_hot_encoder, file)

with open('scaler.pkl', 'wb') as file:
  pickle.dump(scaler, file)

In [49]:
## Define a function to create a model and try with different parameters(Keras Classifier)
def create_model(neurons = 32, layers = 1):
  model = Sequential()
  ## Add I/P Layer
  number_of_input_nodes = X_train.shape[1]
  model.add(Dense(neurons, activation = 'relu', input_shape= (number_of_input_nodes, )))

  ## Add 'layers-1' number of hidden layers HL
  for _ in range(layers-1):
    model.add(Dense(neurons, activation='relu'))

  ## Add O/P Layer
  model.add(Dense(1, activation= 'sigmoid'))

  model.compile(
    optimizer = tf.keras.optimizers.Adam(learning_rate = 0.01),
    loss = tf.keras.losses.BinaryCrossentropy(),
    metrics = ['accuracy'] ## the accuracy we usually use in Classification problems
  )

  return model

In [50]:
## Create a Keras Classifier
model = KerasClassifier(
  build_fn=create_model, ## the function we just created
  layers = 1,
  neurons = 32,
  epochs=50,
  verbose=1  ## to show info logs while running
)

In [51]:
## Define the grid search parameters
param_grid = {
  'neurons' : [16, 32, 64, 128],
  'layers' : [1, 2, 3],
  'epochs' : [50, 100]
}

In [52]:
## Perform Grid Search
grid = GridSearchCV(
  estimator=model,
  param_grid=param_grid,
  n_jobs=-1,
  cv=3,
  verbose=1  ## to show info logs while running
)

In [53]:
grid_result = grid.fit(X_train, y_train)

print("Best: %f using %s" % (grid_result.best_score_, grid_result.best_params_))

Fitting 3 folds for each of 24 candidates, totalling 72 fits


d:\DATA SCIENCE ML AI\DEEP_LEARNING\.venv\Lib\site-packages\scikeras\wrappers.py:915: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)



Epoch 1/50


250/250 [==============================] - 2s 2ms/step - loss: 0.3931 - accuracy: 0.8359
Epoch 2/50
250/250 [==============================] - 0s 2ms/step - loss: 0.3624 - accuracy: 0.8528
Epoch 3/50
250/250 [==============================] - 0s 2ms/step - loss: 0.3576 - accuracy: 0.8576
Epoch 4/50
250/250 [==============================] - 0s 2ms/step - loss: 0.3487 - accuracy: 0.8553
Epoch 5/50
250/250 [==============================] - 0s 2ms/step - loss: 0.3431 - accuracy: 0.8600
Epoch 6/50
250/250 [==============================] - 0s 2ms/step - loss: 0.3443 - accuracy: 0.8566
Epoch 7/50
250/250 [==============================] - 0s 2ms/step - loss: 0.3395 - accuracy: 0.8615
Epoch 8/50
250/250 [==============================] - 0s 2ms/step - loss: 0.3404 - accuracy: 0.8616
Epoch 9/50
250/250 [==============================] - 0s 2ms/step - loss: 0.3380 - accuracy: 0.8585
Epoch 10/50
250/250 [==============================] - 0s 1ms/step - loss: 0.3374 - accuracy: 0.8

## Best: 0.856874 using {'epochs': 50, 'layers': 1, 'neurons': 128}